# Policy Proxy SAE Tutorial

This notebook explains the policy proxy SAE pipeline from first principles.

The goal is to make the project readable at a **high level first**, while still giving the exact formulas and file paths needed to connect the ideas to code and artifacts.


## How to use this notebook

1. Read the markdown cells in order before running the code cells.
2. The code cells are lightweight and mostly inspect configs, manifests, and outputs.
3. Heavy training and large reruns belong on Lambda, not in this notebook.
4. If raw AGORA data or locked benchmark outputs are missing on this machine, the notebook will keep going and show fallback paths.
5. Treat this notebook as a bridge between the paper, the configs, and the saved artifacts.


In [ ]:
from __future__ import annotations

import csv
import json
import platform
import sys
from pathlib import Path

try:
    import yaml
except ImportError:
    yaml = None


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'src').exists() and (candidate / 'configs').exists():
            return candidate
    raise RuntimeError('Could not locate the repository root.')


def read_json_if_exists(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))


def read_csv_rows(path: Path, limit: int = 5) -> list[dict[str, str]]:
    if not path.exists():
        return []
    with path.open('r', encoding='utf-8', newline='') as handle:
        return list(csv.DictReader(handle))[:limit]


def first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


REPO_ROOT = find_repo_root()
print(f'Repository root: {REPO_ROOT}')


In [ ]:
print(f'Python version: {platform.python_version()}')
print(f'Executable: {sys.executable}')
print(f'Platform: {platform.platform()}')
print(f'PyYAML available: {yaml is not None}')


## Tutorial roadmap

We will move through the pipeline in the same order as the paper.

1. Define the task units and notation.
2. Inspect AGORA segments, proxy manifests, and matched negatives.
3. Explain dense representations, SAE features, and proxy-specific feature discovery.
4. Define transfer, robustness, and causal ablation mathematically.
5. Show how the benchmark wrapper aggregates evidence.
6. Show how the same evidence becomes analyst-facing highlighting, retrieval, and triage.
7. Point to the exact saved artifacts and Lambda commands used for final runs.


## 1. Problem setup and notation

The empirical unit is a **policy segment** \(x\), not a whole document.

Each proxy task \(t\) is a binary distinction between positive examples of one policy concept and matched negatives.

In the current benchmark, the six proxy tasks are grouped into three pairs.

1. Bias and Discrimination
2. Privacy and Rights violation
3. Transparency and Interpretability

We use the following notation.

1. \(x_i\) is one policy segment.
2. \(y_i^{(t)} \in \{0,1\}\) is the label for proxy \(t\).
3. \(h_{\ell}(x_i)\) is the pooled dense representation from layer \(\ell\).
4. \(z_{\ell}(x_i)\) is the SAE feature vector at layer \(\ell\).
5. \(F_t\) is the selected sparse feature set for proxy \(t\).
6. \(w_{t,f}\) is the signed weight for feature \(f\) under proxy \(t\).

The project no longer claims a broad family-level shared mechanism. The unit of mechanistic evidence is **proxy-specific sparse features**, with pair-level evidence used as a secondary test.


## 2. Claim boundaries and research questions

The paper keeps a narrow claim boundary.

1. We do **not** claim that sparse SAE is the best overall predictor.
2. We do **not** claim a universal family-level shared mechanism.
3. We do **not** present the assistant as an end-to-end policy judge.

The four main research questions are:

1. **Localization**: Which sparse SAE features reliably localize each proxy?
2. **Stability**: Are those features stable under resampling and masking?
3. **Selective causal support**: Does ablating selected sparse features reduce target proxy evidence more than matched random controls?
4. **Application**: Can these artifacts support analyst-facing highlighting, retrieval, and triage?


In [ ]:
benchmark_cfg_path = REPO_ROOT / 'configs' / 'policy_feature_benchmark_locked_2b.yaml'
benchmark_cfg = yaml.safe_load(benchmark_cfg_path.read_text(encoding='utf-8')) if yaml is not None else None

print(f'Benchmark config: {benchmark_cfg_path}')
if benchmark_cfg is not None:
    print('Pairs in the locked 2B benchmark:')
    for pair in benchmark_cfg['v1_pairs']:
        left = pair['left']['display_name']
        right = pair['right']['display_name']
        print(f"  {pair['family']}: {left} <-> {right}")


## 3. AGORA data and grouped splits

AGORA provides policy text at the segment level, together with metadata and annotations.

The split rule is important: **all segments from the same document must stay in the same split**. This prevents leakage from nearly duplicate text across train, dev, and test.

If \(d(x_i)\) is the document ID of segment \(x_i\), then the split constraint is:

$$
d(x_i) = d(x_j) \Rightarrow \text{split}(x_i) = \text{split}(x_j).
$$

This grouped split rule matters more than a random row-level split because policy documents contain repeated template language.


In [ ]:
candidate_agora_dirs = [
    REPO_ROOT / 'agora' / 'agora',
    REPO_ROOT / 'data' / 'raw' / 'agora',
]
agora_dir = first_existing(candidate_agora_dirs)
print('Candidate AGORA roots:')
for path in candidate_agora_dirs:
    print(f'  {path} -> {path.exists()}')

if agora_dir is None:
    print('No raw AGORA directory found on this machine.')
else:
    print(f'Using AGORA root: {agora_dir}')


## 4. Proxy manifests and matched negatives

The benchmark does not read raw AGORA rows directly during training. It first materializes **proxy manifests**.

For each proxy \(t\), the manifest contains:

1. positive rows for train, dev, and test
2. matched negatives for train, dev, and test
3. validated positives and validated negatives when available

Matched negatives are designed to reduce trivial shortcuts.

At a high level, the matching score combines semantic similarity and metadata compatibility.

$$
\text{MatchScore}(p, n) = \alpha \cdot \text{TextSim}(p, n) + \beta \cdot \text{MetaCompat}(p, n)
$$

where metadata compatibility can include authority, jurisdiction, document form, and other document-level controls.

The practical goal is simple: the negative example should look like a realistic policy segment from a similar setting, but it should not carry the target proxy label.


In [ ]:
manifest_root = REPO_ROOT / 'data' / 'processed' / 'public_values'
manifest_summary_path = manifest_root / 'summary.json'
manifest_summary = read_json_if_exists(manifest_summary_path)

print(f'Manifest root exists: {manifest_root.exists()}')
print(f'Manifest summary exists: {manifest_summary_path.exists()}')
if manifest_summary:
    print('Manifest summary keys:')
    for key in sorted(manifest_summary.keys())[:20]:
        print(f'  {key}')


In [ ]:
diagnostic_candidates = sorted(manifest_root.glob('*/negatives/*/diagnostics.json'))
print(f'Negative diagnostic files found: {len(diagnostic_candidates)}')
if diagnostic_candidates:
    sample_diag_path = diagnostic_candidates[0]
    sample_diag = read_json_if_exists(sample_diag_path)
    print(f'Sample diagnostic path: {sample_diag_path}')
    print(json.dumps(sample_diag, indent=2)[:2000])


## 5. Mechanistic representation primer

The benchmark looks inside the model at a specific **layer** and **site**.

In the current project, the main site is `resid_post`, the residual stream after a transformer block writes its update.

For a segment \(x\):

1. tokenize the forced-choice prompt built from the segment text
2. capture the residual stream sequence at layer \(\ell\)
3. pool across tokens to obtain a segment vector \(h_{\ell}(x)\)
4. optionally encode that vector through the SAE to obtain \(z_{\ell}(x)\)

The SAE turns a dense vector into a sparse feature representation.

$$
z_{\ell}(x) = \mathrm{Enc}(h_{\ell}(x)), \qquad \hat{h}_{\ell}(x) = \mathrm{Dec}(z_{\ell}(x)).
$$

We then treat the nonzero coordinates of \(z_{\ell}(x)\) as candidate mechanistic features.


## 6. Proxy-specific feature discovery and dossiers

For one proxy \(t\), discovery asks which sparse features separate positives from matched negatives.

A linear scoring view is:

$$
m_t(x) = b_t + \sum_{f \in F_t} w_{t,f} z_f(x),
$$

where \(m_t(x)\) is the proxy margin, \(w_{t,f}\) is the signed weight for feature \(f\), and \(F_t\) is the selected feature bank.

A probability-style score is then

$$
s_t(x) = \sigma(m_t(x)).
$$

The notebook and paper use a **feature dossier** abstraction. Each dossier row stores at least:

1. proxy slug
2. family
3. selected layer
4. feature ID
5. signed weight
6. bootstrap stability
7. feature priority
8. top activating segments
9. masking retention at the feature-set level
10. assistant usage count

The current ranking rule is

$$
\mathrm{FeaturePriority}_{t,f} = 0.70 \cdot \mathrm{MinMax}(|w_{t,f}|) + 0.30 \cdot \mathrm{Stability}_{t,f}.
$$

Two additional summary statistics are useful.

1. **Activation gap**
   $$
   \mathrm{ActivationGap}_{t,f} = \mathbb{E}_{x \sim P_t^+}[z_f(x)] - \mathbb{E}_{x \sim P_t^-}[z_f(x)]
   $$
2. **Contribution gap**
   $$
   \mathrm{ContributionGap}_{t,f} = \mathbb{E}_{x \sim P_t^+}[w_{t,f} z_f(x)] - \mathbb{E}_{x \sim P_t^-}[w_{t,f} z_f(x)]
   $$

This notebook's attribution view is **contribution-based**, not gradient-based.


In [ ]:
sparse_cfg_path = REPO_ROOT / 'configs' / 'policy_feature_benchmark_locked_2b.yaml'
sparse_cfg = yaml.safe_load(sparse_cfg_path.read_text(encoding='utf-8')) if yaml is not None else {}
sparse_method = sparse_cfg.get('methods', {}).get('sparse_sae_feature_bank', {})
print('Sparse discovery settings:')
for key in ['candidate_layers', 'site', 'pooling', 'robustness_pooling', 'topk', 'discovery_overlap_reseeds']:
    print(f'  {key}: {sparse_method.get(key)}')


## 7. Stability and robustness

Discovery alone is not enough. The benchmark asks whether the selected features survive stress.

### 7.1 Bootstrap stability

Each feature gets a selection-frequency style stability score under repeated resampling.

$$
\mathrm{Stability}_{t,f} = \frac{1}{B} \sum_{b=1}^{B} \mathbf{1}[f \in F_t^{(b)}].
$$

High stability means the feature keeps reappearing when the data are perturbed.

### 7.2 Masking robustness

Let \(\mathrm{AUC}_t\) be the held-out proxy AUROC before masking and \(\mathrm{AUC}^{\mathrm{mask}}_t\) after masking proxy keywords. The retention ratio is

$$
\mathrm{Retention}_t = \frac{\mathrm{AUC}^{\mathrm{mask}}_t}{\mathrm{AUC}_t}.
$$

The benchmark clips this value into \([0,1]\) for the robustness axis:

$$
\mathrm{RobustnessTaskScore}_t = \mathrm{clip}(\mathrm{Retention}_t, 0, 1).
$$

Retention greater than 1 can still occur in the raw diagnostics. That does not mean masking helped in a causal sense. It usually means the masked prompt distribution changed in a way that happened to favor the classifier.


## 8. Pair-linked transfer

Transfer tests whether a feature bank discovered for one proxy helps on its paired proxy more than on unrelated controls.

For a directed comparison \(a \rightarrow b\):

1. train or select the feature bank on source proxy \(a\)
2. evaluate the bank on the target proxy \(b\)
3. compare that transfer AUROC to the mean AUROC on cross-family controls

The main quantity is the directed consistency gap:

$$
\mathrm{ConsistencyGap}_{a \rightarrow b} = \mathrm{AUC}_{a \rightarrow b} - \frac{1}{|C(a)|} \sum_{c \in C(a)} \mathrm{AUC}_{a \rightarrow c},
$$

where \(C(a)\) is the set of cross-family control tasks for source proxy \(a\).

A positive gap supports **pair-linked support**. It does **not** by itself prove that the exact same sparse feature is reused mechanistically.


## 9. Targeted proxy feature-set ablation

The causal step is deliberately narrow.

1. choose one proxy \(t\)
2. choose the top \(k\) discovered sparse features for that proxy
3. zero those SAE latents at one layer and one site during a full forward pass
4. compare the target-label margin drop to matched random feature sets of the same cardinality and layer

In code, the editor zeroes selected SAE latents and decodes the edited vector back into the residual stream at the chosen layer-site.

Let the forced-choice prompt compare the target label to a contrast label. The baseline and ablated margins for one segment \(x_i\) are

$$
m_i^{\mathrm{base}} = \log p(y_t \mid x_i) - \log p(y_c \mid x_i),
$$

$$
m_i^{\mathrm{ablate}} = \log p(y_t \mid x_i; \text{ablated } F_{t,k}) - \log p(y_c \mid x_i; \text{ablated } F_{t,k}).
$$

The target margin drop is

$$
\Delta_i^{\mathrm{target}} = m_i^{\mathrm{base}} - m_i^{\mathrm{ablate}}.
$$

For matched random controls \(R_{t,k}^{(r)}\) at the same layer and cardinality,

$$
\Delta_i^{\mathrm{random}} = \frac{1}{R} \sum_{r=1}^{R} \left(m_i^{\mathrm{base}} - m_i^{\mathrm{ablate},(r)}\right).
$$

The per-example paired delta is

$$
\delta_i = \Delta_i^{\mathrm{target}} - \Delta_i^{\mathrm{random}}.
$$

Finally, the proxy-level selective causal effect is

$$
\mathrm{CausalSelectivity}_{t,k} = \frac{1}{N} \sum_{i=1}^{N} \delta_i.
$$

The current pipeline marks a proxy as causally supported only when the selectivity is positive and the lower confidence bound is above zero.


## 10. Benchmark aggregation and headline metrics

The benchmark keeps three main axes for all methods.

### 10.1 Coverage

$$
\mathrm{CoverageScore} = \frac{1}{|T|} \sum_{t \in T} \mathrm{AUC}_t.
$$

### 10.2 Robustness

$$
\mathrm{RobustnessScore} = \frac{1}{|T|} \sum_{t \in T} \mathrm{clip}\left(\frac{\mathrm{AUC}^{\mathrm{mask}}_t}{\mathrm{AUC}_t}, 0, 1\right).
$$

### 10.3 Consistency

$$
\mathrm{ConsistencyScore} = \frac{1}{|D|} \sum_{(a \rightarrow b) \in D} \mathrm{ConsistencyGap}_{a \rightarrow b}.
$$

### 10.4 Core score

The current locked config uses equal weights across the three axes.

$$
\mathrm{CoreScore} = \frac{1}{3} \left(\mathrm{CoverageScore} + \mathrm{ConsistencyScore} + \mathrm{RobustnessScore}\right).
$$

Sparse-only causal evidence is summarized separately. It is not mixed into the all-method comparison table.


## 11. Assistant layer and analyst-facing objects

The assistant is a downstream application layer built on top of benchmark artifacts.

It produces three kinds of behavior.

1. **Highlighting**: rank segments within each family by analyst relevance.
2. **Retrieval**: retrieve related policy passages from other documents.
3. **Triage**: prioritize what a human should read first.

For sparse methods, a segment card can expose

1. top feature IDs
2. top feature weights
3. top feature contributions
4. mean feature stability
5. selected layer
6. causal badge

This is the bridge from mechanistic artifacts to analyst-facing support.


## 12. Assistant metrics

The assistant summary uses family-level averages before computing the final overall score.

### 12.1 Highlighting

For each family \(f\), the highlight family score is

$$
H_f = \mathrm{mean}\left(\mathrm{AUROC}, \mathrm{AUPRC}, P@3, R@3, \min(1, 1 / \mathrm{FRR})\right).
$$

Then

$$
\mathrm{HighlightScore} = \frac{1}{|F|} \sum_f H_f.
$$

### 12.2 Retrieval

For each family \(f\), define

$$
R_f = \mathrm{mean}\left(\mathrm{Hit@5}, \mathrm{MRR@5}, \mathrm{nDCG@5}, \mathrm{clip}\left(\frac{\mathrm{WithinMinusCrossRate}+1}{2}, 0, 1\right)\right).
$$

Then

$$
\mathrm{RetrievalScore} = \frac{1}{|F|} \sum_f R_f.
$$

### 12.3 Triage

For each family \(f\), define

$$
T_f = \mathrm{mean}\left(R@3, \min(1, 1 / \mathrm{FirstRelevantRank}), \min(1, 1 / \mathrm{AverageReviewDepth}), \mathrm{nDCG@5}\right).
$$

Then

$$
\mathrm{TriageScore} = \frac{1}{|F|} \sum_f T_f.
$$

The final assistant score is

$$
\mathrm{AssistantCoreScore} = \frac{1}{3} \left(\mathrm{HighlightScore} + \mathrm{RetrievalScore} + \mathrm{TriageScore}\right).
$$


In [ ]:
result_root_candidates = [
    REPO_ROOT / 'results' / 'policy_feature_benchmark_locked_2b',
    REPO_ROOT / 'r' / 'bf2',
]
assistant_root_candidates = [
    REPO_ROOT / 'results' / 'policy_analysis_assistant_locked_2b',
    REPO_ROOT / 'r' / 'as2',
]
benchmark_root = first_existing(result_root_candidates)
assistant_root = first_existing(assistant_root_candidates)
mech_fig_root = first_existing([
    REPO_ROOT / 'analysis' / 'mech_interp_v2' / 'figures',
])

print(f'Benchmark root: {benchmark_root}')
print(f'Assistant root: {assistant_root}')
print(f'Mechanistic figure root: {mech_fig_root}')


In [ ]:
interesting_files = []
if benchmark_root is not None:
    interesting_files.extend([
        benchmark_root / 'summary' / 'table_main_benchmark_results.csv',
        benchmark_root / 'summary' / 'table_proxy_mechanistic_evidence.csv',
        benchmark_root / 'summary' / 'table_pair_transfer_and_causality.csv',
        benchmark_root / 'summary' / 'proxy_feature_summary.csv',
        benchmark_root / 'summary' / 'proxy_causal_summary.csv',
        benchmark_root / 'summary' / 'feature_concept_cards.jsonl',
    ])
if assistant_root is not None:
    interesting_files.extend([
        assistant_root / 'summary' / 'assistant_leaderboard.json',
        assistant_root / 'summary' / 'assistant_feature_usage.csv',
        assistant_root / 'summary' / 'assistant_feature_cards.jsonl',
    ])

for path in interesting_files:
    print(f'{path} -> {path.exists()}')


In [ ]:
preview_targets = []
if benchmark_root is not None:
    preview_targets.append(benchmark_root / 'summary' / 'table_proxy_mechanistic_evidence.csv')
    preview_targets.append(benchmark_root / 'summary' / 'proxy_causal_summary.csv')
if assistant_root is not None:
    preview_targets.append(assistant_root / 'summary' / 'assistant_feature_usage.csv')

for path in preview_targets:
    print('\n' + '=' * 80)
    print(path)
    rows = read_csv_rows(path, limit=5)
    if not rows:
        print('No rows found or file missing.')
    else:
        for row in rows:
            print(row)


## 13. Reading the saved artifacts

Once a benchmark run is complete, the most important files are usually:

1. `table_main_benchmark_results.csv` for all-method comparison
2. `table_proxy_mechanistic_evidence.csv` for proxy-level sparse evidence
3. `table_pair_transfer_and_causality.csv` for directed pair-linked evidence
4. `proxy_feature_summary.csv` for per-proxy sparse feature statistics
5. `proxy_causal_summary.csv` for ablation summaries
6. `feature_concept_cards.jsonl` for top activating examples and feature semantics
7. `assistant_feature_cards.jsonl` and `assistant_feature_usage.csv` for analyst-facing linkage

A good reading order is:

1. overall benchmark table
2. proxy mechanistic table
3. causal summary
4. pair table
5. assistant leaderboard
6. assistant cards and feature concept cards


## 14. Representative policy analyses

The appendix in the paper uses recognizable policy sources, not internal document IDs alone.

For each case study, the intended readout is:

1. source policy name
2. quoted or excerpted policy passage
3. surfaced proxy
4. selected layer
5. top feature IDs
6. mean feature stability
7. causal badge

That presentation is important because the paper is not only about benchmark numbers. It is also about whether mechanistic artifacts can be rendered into a usable evidence chain for analysts.


In [ ]:
login_command = "export HF_TOKEN='PASTE_YOUR_TOKEN_HERE' && python -c \"from huggingface_hub import login; import os; login(token=os.environ['HF_TOKEN']); print('HF login complete')\""
access_check_command = "python -c \"from transformers import AutoTokenizer; AutoTokenizer.from_pretrained('google/gemma-2-2b'); print('Gemma access OK')\""
locked_2b_command = "cd ~/agora-mi && source .venv/bin/activate && python scripts/run_policy_feature_benchmark.py --preflight_only --config configs/policy_feature_benchmark_locked_2b.yaml --output_root results/policy_feature_benchmark_locked_2b && python scripts/run_policy_feature_benchmark.py --config configs/policy_feature_benchmark_locked_2b.yaml --output_root results/policy_feature_benchmark_locked_2b && python scripts/aggregate_policy_feature_benchmark.py --config configs/policy_feature_benchmark_locked_2b.yaml --output_root results/policy_feature_benchmark_locked_2b && python scripts/run_dense_subspace_control_eval.py --config configs/policy_mech_interp.yaml --manifest_root data/processed/public_values --family equality_neutrality --proxy_slug risk_factors_bias --paired_proxy_slug harms_discrimination --layer 20 --site resid_post --split test --max_samples 100 --random_sets 100 --k 3 && python scripts/run_dense_subspace_control_eval.py --config configs/policy_mech_interp.yaml --manifest_root data/processed/public_values --family individual_rights --proxy_slug risk_factors_privacy --paired_proxy_slug harms_violation_of_civil_or_human_rights_including_privacy --layer 24 --site resid_post --split test --max_samples 100 --random_sets 100 --k 3 && python scripts/run_policy_analysis_experiments.py --config configs/policy_analysis_assistant_locked_2b.yaml --output_root results/policy_analysis_assistant_locked_2b && printf '%s\n' 'The agency shall publish an annual transparency report describing system deployment, audit procedures, and material incidents. Any system processing personal data must include retention limits, access controls, and independent review. Operators should assess whether deployment could create discriminatory effects for protected groups and document mitigation steps before rollout.' > sample_policy_doc.txt && python scripts/run_policy_document_analysis.py --input_path sample_policy_doc.txt --config configs/policy_analysis_assistant_locked_2b.yaml --output_path results/policy_document_analysis_locked_2b.json --document_id lambda_doc_1 --title 'Lambda Document' --source_type lambda_text"
locked_9b_command = "cd ~/agora-mi && source .venv/bin/activate && python scripts/run_policy_feature_benchmark.py --preflight_only --config configs/policy_feature_benchmark_locked_9b_primary.yaml --output_root results/policy_feature_benchmark_locked_9b_primary && python scripts/run_policy_feature_benchmark.py --config configs/policy_feature_benchmark_locked_9b_primary.yaml --output_root results/policy_feature_benchmark_locked_9b_primary && python scripts/aggregate_policy_feature_benchmark.py --config configs/policy_feature_benchmark_locked_9b_primary.yaml --output_root results/policy_feature_benchmark_locked_9b_primary"

print('HF login command:\n')
print(login_command)
print('\nGemma access check:\n')
print(access_check_command)
print('\nLocked 2B package:\n')
print(locked_2b_command)
print('\nLocked 9B primary replication:\n')
print(locked_9b_command)


## 15. Final interpretation checklist

When reading or presenting results, keep this interpretation ladder.

1. **Localization**: Did we identify sparse features aligned to the proxy?
2. **Stability**: Do those features persist under resampling and masking?
3. **Selective causal support**: Does ablating them outperform matched random controls?
4. **Pair-linked support**: Is transfer positive relative to cross-family controls?
5. **Assistant rendering**: Can the same artifacts support highlighting, retrieval, and triage?

The intended paper claim is narrow.

1. Sparse SAE features do not need to win the aggregate benchmark.
2. They need to provide localized, stress-tested, and selectively causal evidence for some proxies.
3. The assistant layer should be presented as analysis support, not as a policy judge.

If you remember only one idea from this notebook, remember this one:

**the project is about turning sparse internal features into auditable evidence objects for policy analysis.**
